# Notebook 5: Building HTML Visualization Booklets

In this notebook, we will take the outputs of both our disambiguation and our dependency modules, and build a booklet where we can see a collection of before and after parses for each module. 

Using these booklets to visualize outputs is very useful for understanding the data, since we can see the outputs right away, compare Ojibwe sentence to their English counterparts, and search for sentences with certain attributes (i.e., sentences containing a `PRONDem` demonstrative).

In [17]:
# setup paths
from __future__ import annotations
from pathlib import Path
import sys, os

CWD = Path.cwd().resolve()
REPO_ROOT = Path(os.getcwd()).resolve().parents[1]
SRC_DIR = REPO_ROOT / "src"
if str(REPO_ROOT) not in sys.path:
    sys.path.append(str(REPO_ROOT))
if str(SRC_DIR) not in sys.path: 
    sys.path.append(str(SRC_DIR))

# inputs
OJ_PATH = REPO_ROOT / "data" / "parallel" / "oj_50.txt"
EN_PATH = REPO_ROOT / "data" / "parallel" / "en_50.txt"

# Grammars and FST
DISAMBIG_CG_PATH = REPO_ROOT / "data"/ "grammars" / "disambiguation.cg3"
DEPENDENCY_CG_PATH = REPO_ROOT / "data"/ "grammars" / "dependency.cg3"
FST_PATH = REPO_ROOT /"data" / "fst" / "ojibwe7.fomabin"

# Output folder for HTML booklets
BOOKLETS_DIR = REPO_ROOT / "data" / "booklets"
BOOKLETS_DIR.mkdir(parents=True, exist_ok=True)

# Output files
DISAMBIG_HTML = BOOKLETS_DIR / "disambig_50.html"
DEP_HTML = BOOKLETS_DIR / "dep_50.html"

# treebank file (parsed in notebook #4, but we will reparse)
TREEBANK_PATH = REPO_ROOT / "data" / "treebanks" / "oj_50_treebank.conllu"

for p in [OJ_PATH, EN_PATH]:
    if not p.is_file():
        raise FileNotFoundError(f"Missing: {p}")

### Parsing the disambiguation booklet

We will first start off by disambiguated the sentences in `oj_50.txt`, and then making a booklet of the before / after disambiguation CG3. For this, we will also need to input the corresponding 50 English sentences in `en_50.txt`.

In [4]:
from treebank_modules.booklets import build_disambig_booklet
from grammar_modules.fst import Fst

# start by buiding the disambiguation booklet on the 50 sentences
build_disambig_booklet(
    ojibwe_path=OJ_PATH,
    english_path=EN_PATH,
    cg3_grammar_path=DISAMBIG_CG_PATH,
    fst=Fst(FST_PATH),
    out_html_path=DISAMBIG_HTML,
    html_title="Ojibwe Disambiguation Booklet"
)


Output()

FST file is /home/ktu/dev/ling/Ojibwe_Constraint_Grammar/data/fst/ojibwe7.fomabin


✔ Disambiguation booklet written to /home/ktu/dev/ling/Ojibwe_Constraint_Grammar/data/booklets/disambig_50.html (50
sentences)

### Inspecting the output

Now we can totally open the HTML booklet in `data/booklets/` with a live server, but just for expository purposes, we can quickly peek at the first output.

Below we will be able to see the before and after disambiguation for the first 2 blocks. While the first sentence *Odaanaang bimibatoowan odayan gaa-bimaagonebizod.* isn't fully disambiguated, we do see that a `2SgSubj` reading for the verb *bimibatoowan* gets removed. In the second sentence, we see a fully disambiguated sentence, where *gii-pimi-ayaagwen* keeps the best readings based on (1) the surrounding noun agreeing `3SgProxSubj` and (2) the lemma being lexicalized (`bimi-ayaa` instead of `ayaa`).

In [ ]:
# display the first few HTML sentences
from IPython.display import HTML, display
from pathlib import Path

# how many sents to show
num_sents = 2

html = Path(DISAMBIG_HTML).read_text(encoding="utf-8")
css  = f"<style>figure:nth-of-type(n+{num_sents+1}){{display:none;}}</style>"
html_lite = html.replace("<body>", f"<body>{css}", 1)

display(HTML(html_lite))



### Building the dependency booklet

Now, we will build the dependency booklet on the same subset of 50 sentences. We will go from scratch, inputting both the Ojibwe and English sentences, parsing dependencies, converting go CoNLL-U format, and finally making the visualization booklet.

As a side note, we technically already built a CoNLL-U formatted treebank for these sentences in the `04_corpus.ipynb` notebook, so if we wanted to see the dependency arc visualizations (no CG3 output), then we could set `reparse_with_cg3=False`. Although, normally having the CG3 output alongside the dependency visualization is useful for debugging, so in the following block we will stick to `reparse_with_cg3=True`. 

In [ ]:
from src.booklets import build_dep_booklet

try:
    import pyconll, spacy  # just to give an immediate error if missing
except Exception as e:
    print("need to pip install pyconll spacy")
    raise

build_dep_booklet(
    treebank_path=TREEBANK_PATH,
    ojibwe_path=OJ_PATH,
    english_path=EN_PATH,
    out_html_path=DEP_HTML,
    html_title="Ojibwe Dependency Booklet",
    reparse_with_cg3=True,
    disamb_grammar_path=DISAMBIG_CG_PATH,
    dep_grammar_path=DEPENDENCY_CG_PATH,
    fst_path=FST_PATH
)


### Inspecting the output

Taking a look at the first 2 blocks of the dependency booklet below, we can now see the dependency visualizations of the sentences, along with their CG3 outputs. This is useful for seeing directly how the CG3 output maps to the dependency arcs, and is a useful tool for both development and data exploration.

In [ ]:
from IPython.display import HTML, display
from pathlib import Path

# how many sents to display
num_sents = 2

html = Path(DEP_HTML).read_text(encoding="utf-8")
css  = f"<style>figure:nth-of-type(n+{num_sents+1}){{display:none;}}</style>"
html_lite = html.replace("<body>", f"<body>{css}", 1)

display(HTML(html_lite))



### Summary of the `booklets.py` module

To summarize, the booklets provide a simple way to view the outputs of both the disambiguation and the dependency parsing modules. These booklets are useful for development, as they allow for easy visualization of data.

Also, although not shown above, we can also preprocess data that we send into these booklets, for example preprocessing a list of all sentences that contain `VTA` verbs, or sentences that contain originally contain no ambiguities.  

